In [7]:
import json

from openai import OpenAI
from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents

openai_client = OpenAI()

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

def search(query):
    return index.search(query=query, num_results=5)

def llm_structured(user_prompt, instructions, output_format):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_format
    )

    return response.output_parsed

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 385 chunks from 95 documents


We have our RAG response model with structured output:

In [8]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question")
    found_answer: bool = Field(description="True if relevant information was found")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="Category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

And the RAG function that combines everything:

In [9]:
def rag(query, output_format=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(prompt, instructions, output_format)

## The Problem with Text Search

Text search looks at exact words. Consider these two questions:

- “install Evidently locally”
- “how can I set up Evidently in my projects?”

They mean the same thing but use different words (“install” vs “set up”). If the documentation uses “installation” and the user asks about “setup,” text search won’t find a match.

**The issue:** text search doesn’t understand meaning — it only matches words.

---

## How Vector Search Works

Vector search matches meaning instead of exact words by turning text into numbers (vectors).

- Text is converted to a high-dimensional vector (e.g., 384 or 768 numbers).
- Similar texts end up with similar vectors.
- The distance between vectors indicates semantic similarity.

**At search time:**

1. Convert the query to a vector.  
2. Compare it against all document vectors.  
3. Return the most similar documents.

This is also called **semantic search** because it searches based on meaning, not words.

---

For a deeper introduction to embeddings and vector search, see the **Build Your Own Search Engine** workshop.

In [10]:
!uv add sentence-transformers

Resolved 169 packages in 13.79s
 Downloaded networkx
 Downloaded tokenizers
 Downloaded hf-xet
 Downloaded transformers
 Downloaded torch
Prepared 12 packages in 4m 12s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 12 packages in 9.02s
 + filelock==3.20.3
 + fsspec==2026.2.0
 + hf-xet==1.2.0
 + huggingface-hub==1.4.1
 + networkx==3.6.1
 + safetensors==0.7.0
 + sentence-transformers==5.2.2
 + shellingham==1.5.4
 + tokenizers==0.22.2
 + torch==2.10.0
 + transformers==5.1.0
 + typer-slim==0.21.1
Resolved 169 packages in 1ms
Audited 147 packages in 205ms


In [11]:
from sentence_transformers import SentenceTransformer

In [12]:
model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\AI-Engineering\AI-Engineering-Buildcamp\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\victus\.cache\huggingface\hub\models--sentence-transformers--multi-qa-MiniLM-L6-cos-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Why *multi-qa-MiniLM-L6-cos-v1*?

- **Small and fast** — smaller models run faster.
- **Optimized for QA** — built for question-answering tasks.
- **Cosine similarity** — great for semantic similarity.

---

## How to Choose a Pretrained Model

**Key factors:**

- **Size**  
    Smaller models are faster but may be less accurate. For learning and prototyping, a smaller model is usually fine.

- **Similarity function**  
    Models with **`cos`** in the name use **cosine similarity**. Others typically use **dot product**.

- **Task**  
    Some models are tuned for **QA**, **clustering**, or **semantic search**.

---

If you’re unsure about similarity metrics, ask for a quick explanation of **cosine similarity vs dot product vs Euclidean distance**.

In [13]:
q1 = 'install Evidently locally'
q2 = 'how can I set up evidently in my projects?'


In [14]:
v1 = model.encode(q1)
v1[:20]


array([ 0.07184875, -0.02879699,  0.06123495, -0.01810761,  0.12533103,
       -0.04134639, -0.06354983, -0.08986846, -0.0414282 , -0.04465347,
        0.06421647,  0.02077092,  0.00219163,  0.00508835, -0.00833955,
       -0.03770018,  0.03059414, -0.04844624,  0.06848747, -0.01030118],
      dtype=float32)

In [15]:
v1.shape

(384,)

In [16]:
v2 = model.encode(q2)

In [17]:
v1.dot(v2)

np.float32(0.4347795)

The score is much lower (around 0.02). The higher the dot product, the more similar the texts are.

## Creating Document Embeddings

 

Now let's create embeddings for all our documents. First, prepare the text:

 

In [18]:
texts = []

for doc in chunked_docs:
    title = doc.get('title', '')
    description = doc.get('description', '')
    content = doc.get('content', '')

    text = title + " " + description + " " + content
    texts.append(text.strip())


In [ ]:
embeddings = model.encode(texts, show_progress_bar=True)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
#This creates a matrix with rows (documents) and columns (dimensions):
embeddings.shape

In [22]:
embeddings

array([[-0.0484977 ,  0.06601195,  0.01094064, ...,  0.03090534,
         0.06879062,  0.05070845],
       [-0.03232048, -0.01947779, -0.01677326, ...,  0.01519901,
         0.02379611, -0.00774647],
       [-0.02108491, -0.0565795 ,  0.00483409, ..., -0.04806893,
         0.0710432 ,  0.04317408],
       ...,
       [-0.00160592, -0.04139434, -0.05660459, ...,  0.05727456,
         0.09037897, -0.02732164],
       [-0.02243868, -0.00113358, -0.05355776, ...,  0.04080839,
         0.08891257, -0.00143775],
       [-0.06048303, -0.05174242,  0.01422585, ...,  0.00431389,
         0.09889208,  0.00393773]], shape=(385, 384), dtype=float32)

## Vector Search with `minsearch`

We can use dot product to find similar documents. Let's multiply our query vector by all document vectors:

```python
scores = embeddings.dot(v1)
scores[:15]
```

The scores show similarity. Higher scores mean more similar documents. We can sort by score to get the most relevant results.

`minsearch` has a `VectorSearch` class that handles this. First import it:

```python
from minsearch import VectorSearch
```

In [23]:
from minsearch import VectorSearch

In [24]:
vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(embeddings, chunked_docs)

Now create a search function that encodes the query:

In [25]:
def vector_search(query, num_results=5):
    vector = model.encode(query)
    return vindex.search(vector, num_results=num_results)

In [26]:
def rag_vector(query, output_format=RAGResponse):
    search_results = vector_search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(prompt, instructions, output_format)


In [28]:
query='LLM as a Judge'

In [29]:
answer = rag_vector(query)

print(f"Query: {query}")
print(f"found_answer: {answer.found_answer}")
print(f"confidence: {answer.confidence}")
print(f"answer: {answer.answer[:200]}...")

Query: LLM as a Judge
found_answer: True
confidence: 0.9
answer: Using an LLM (Large Language Model) as a judge involves evaluating text responses against custom criteria. There are two primary methods for this evaluation:

1. **Reference-based Evaluation**: This m...


The vector search finds semantically similar documents, even if they don't contain the exact words from the query.

This runs both searches and combines the results. You could add deduplication logic to avoid returning the same document twice (left as an exercise).

 

In [31]:
def rag_hybrid(query, output_format=RAGResponse):
    search_results = hybrid_search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(prompt, instructions, output_format)


In [32]:
query = 'llm as a judge'

In [33]:
results = rag_hybrid(query)

In [34]:
print(results)

answer='An LLM (Large Language Model) can be used as a judge in text evaluation through two primary methods: Reference-based and Open-ended evaluation.\n\n1. **Reference-based Evaluation**: This method compares new responses against a predetermined reference, which can serve as a "ground truth". This is particularly useful for regression testing or when there is a need to determine the correctness of responses based on known accurate outputs.\n\n2. **Open-ended Evaluation**: In this approach, responses are assessed based on custom-defined criteria, allowing for evaluation of outputs when no reference is available.' found_answer=True confidence=0.9 answer_type='explanation' followup_questions=['What are the criteria for using LLM as a judge?', 'How can I implement an LLM judge in my project?', 'What tools are needed to run LLM as a judge?']


## When to Use Vector Search

Vector search adds complexity:

- Extra dependencies (e.g., PyTorch, `sentence-transformers`)
- One-time document encoding
- Per-query encoding latency

### So when is it worth it?

Start with text search. It’s simpler and faster. Introduce vector search only when text search fails, for example:

- Users phrase questions differently than the documentation
- You need semantic understanding
- Documents use varied terminology

Later in the course, we’ll cover evaluation techniques to measure whether vector search improves your RAG system.

> Vector search is a tool, not a requirement. Use it when it adds value.